# Objectif 

Analyse de la portion 1 extraite via le code acouplane_select_and_extract_portion1.ipynb

Date de création : 07/10/2026

In [ ]:
import os
import sys
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from scipy.io import loadmat
from matplotlib.lines import Line2D
from datetime import datetime, timedelta

In [ ]:
if os.name == "nt":  # Windows
    ROOT_PROGRAM = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd"
else:  # Linux
    ROOT_PROGRAM = "/home/program/ubf_tools"
sys.path.append(ROOT_PROGRAM)

from misc import equivalent_celerity
from publication.publication_figure import set_subfigures_abc_labels, color
from real_data_analysis.acouplane.src.acouplane_cst import *
from real_data_analysis.acouplane.src.acouplane_utils import *


# Chargement des données 

Les données sont stockées au format netcdf sous la forme d'un arbre (xarray DataTree) et regroupe l'ensemble des données associées à la portion d'intérêt. 

In [ ]:
portion_id = 1
fpath = f"{ACOUPLANE_PORTION_NC_ROOT_FPATH}_{portion_id}.nc"
portion = xr.open_datatree(fpath)
portion

# Rappel de la position de la portion 

In [ ]:
plot_portion_tir_pos(portion_tree=portion)

# Correction de la position des OBS

In [ ]:
apply_obs_pos_correction = True 

if apply_obs_pos_correction:
    apply_rcv_position_correction_to_portion(portion_tree=portion)
    portion
    plot_portion_tir_pos(portion_tree=portion)

# Visualisation des données de pression

## Signaux

In [ ]:
# Plot 
single_fig = False
for name, node in portion.pressure.children.items():
    ds_pressure_section = node.to_dataset()
    fig, axs = plot_pressure_section(ds_pressure_section=ds_pressure_section, single_fig=single_fig)
    fig.suptitle(f"{ds_pressure_section.section_orientation} {ds_pressure_section.section_id}")
    plt.show()

## Spectrogrames 

**Remarque** : l'amplitude des cannons décroit avec le temps avant de revenir à l'amplitude de départ. Ce comportement nominal des cannons à air comprimé se traduit par une modulation d'amplitude des tirs visibles sur les spectrogrames ci dessous.    

In [ ]:
# Plot 
for name, node in portion.pressure.children.items():
    ds_pressure_section = node.to_dataset()
    fig, axs = plot_spectrogram_section(ds_pressure_section, nperseg=2**8, noverlap=2**7, cmap="magma")
    fig.suptitle(f"{ds_pressure_section.section_orientation} {ds_pressure_section.section_id}")
    plt.show()

# Calcul des grandeurs signantes 

In [ ]:
portion

## Extraction de chaques émissions

### Calcul de la célérité équivalente 

In [ ]:
celerity = xr.open_datatree(ACOUPLANE_CELERITY_NC_FPATH)
celerity

In [ ]:
selected_profile = "T7_00003"
# selected_profile = "T0_00020"

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
for name, node in celerity.children.items():
    if name == selected_profile:
        ds = node.to_dataset()
        ds.celerity.plot(y="depth", ax=ax, yincrease=False, label=name)

    # ds = node.to_dataset()
    # ds.celerity.plot(y="depth", ax=ax, yincrease=False, label=name)


# for ax in axs.flatten():
ax.legend(loc="lower right", ncols=2, fontsize=10)
# axs[1].legend(loc="upper right", ncols=2, fontsize=10)
# axs[1].set_ylabel("")

# plt.ylim(500, 0)

In [ ]:
ds_ = ds.sel(depth=slice(0, 100))
c_eq = equivalent_celerity(ds_.celerity.values, ds_.celerity.depth.values)
c_eq

### Calcul des temps de propagation associés à chaque couple émission / récepteur 

In [ ]:
c0 = c_eq
for name, node in portion.position.children.items():
    ds_pos_section = node.to_dataset()
    # Propagation distance assuming strait line propagation path
    dpropa_direct = np.sqrt(ds_pos_section.src_rcv_geodesic_dist**2 + ds_pos_section.rcv_immersion**2)
    # Compute propagation time 
    tps_propagation_s = dpropa_direct / c0 
    # tps_propagation_s = ds_pos_section.src_rcv_geodesic_dist / c0 
    ds_pos_section["tps_propagation_s"] = tps_propagation_s

    # Derive associated arrival dt 
    tps_propagation_ns = (tps_propagation_s * 1e9).round()
    tps_propagation_td = tps_propagation_ns.astype("timedelta64[ns]")
    ds_pos_section["tir_reception_datetime"] = tps_propagation_td + ds_pos_section.src_datetime

    # Update tree
    node.ds = ds_pos_section

In [ ]:
ds_pos_section.tps_propagation_s

In [ ]:
portion

In [ ]:
(portion.position.sn0.rcv_immersion, portion.position.sn0.rcv_id)

### Comparaison aux signaux 

In [ ]:
# r = ds_pos_section.src_rcv_geodesic_dist.max()
# delta_c = 10
# delta_t = (r*delta_c) / c0**2
# delta_t

c0 = 1508
delta_r = 200
delta_t = 1 / c0 * delta_r
delta_t


In [ ]:
nb_tir = 1
offset1 = 50
offset2 = 3000

obs_ids = list(portion.pressure.children.items())[0][1].to_dataset().obs_id.values
# obs_ids = obs_ids[[0, 3, 5]]
for name, node in portion.pressure.children.items():
    ds_pressure_section = node.to_dataset()  
    # Associated ds_pos
    ds_pos_section = portion.position[name].to_dataset()
    tir_dt = ds_pos_section["tir_reception_datetime"].isel(src_datetime=slice(0, 0+nb_tir))

    fig, axs = plt.subplots(nrows=obs_ids.size, ncols=1, figsize=(16, 14), sharex=False)        
    axs = axs.flatten()
    for i, rcv_id in enumerate(obs_ids):
        tir_dt_rcv = tir_dt.sel(rcv_id=rcv_id)
        # Select associated data window 
        press_rcv = ds_pressure_section[f"pressure_{rcv_id}"].sel({f"time_{rcv_id}":slice(tir_dt_rcv.values[0]-np.timedelta64(offset1, 'ms'), tir_dt_rcv.values[-1]+np.timedelta64(offset2, 'ms'))})
        # Plot pressure signal 
        # Normalize 
        press_rcv = press_rcv / press_rcv.max()
        press_rcv.plot(label=rcv_id.upper(), color=color(i), ax=axs[i])
        # Plot arrivals time 
        axs[i].vlines(x=tir_dt_rcv, ymin=-1, ymax=1, color="k", linestyle="--")
        # Add legend
        axs[i].legend(loc="upper right")

    xlab = axs[0].get_xlabel()
    ylab = axs[0].get_ylabel()

    for ax in axs:
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")

    fig.suptitle(f"{ds_pressure_section.section_orientation} {ds_pressure_section.section_id}")
    fig.supxlabel(xlab)
    fig.supylabel(ylab)

    plt.show()

# Calcul des grandeurs signantes 

In [ ]:
derive_portion_cpsd(
    portion_tree=portion, offset_ms=100, signal_duration_s=3, nfft=2**15, obs_ids=None
)
portion

In [ ]:
ref_rcv_id = "obs1"

# print(axs.shape)
for name, node in portion.feature.children.items():
    ds_feature_section = node.to_dataset()

    ds_feature_section = ds_feature_section.isel(src_datetime=0)

    sjj = ds_feature_section.sel(rcv_i=ref_rcv_id, rcv_j=ref_rcv_id).Sxy
    # Per convention : 2nd index = ref
    fig, axs = plt.subplots(nrows=obs_ids.size, ncols=2, figsize=(16, 8), sharex=True, sharey="col")

    for i_rcv, rcv_id_i in enumerate(obs_ids):

        # Sii
        sii = ds_feature_section.sel(rcv_i=rcv_id_i, rcv_j=rcv_id_i).Sxy
        # Sij
        sij = ds_feature_section.sel(rcv_i=rcv_id_i, rcv_j=ref_rcv_id).Sxy

        # RTF
        pi_ij = sij.values / sjj.values
        gamma_ij = 20 * np.log10(np.abs(pi_ij))
        # np.abs(pi_ij).plot(ax=axs[i_rcv, 0])
        # gamma_ij.plot(ax=axs[i_rcv, 0])
        axs[i_rcv, 0].plot(sij.freq, gamma_ij)

        # DCF (sxy SCOT)
        dcf_ij = sij.values / np.sqrt(sii.values * sjj.values)
        # np.real(dcf_ij).plot(ax=axs[i_rcv, 1], color=color(0))
        # np.imag(dcf_ij).plot(ax=axs[i_rcv, 1], color=color(1))
        axs[i_rcv, 1].plot(sij.freq, np.real(dcf_ij), color=color(0))
        axs[i_rcv, 1].plot(sij.freq, np.imag(dcf_ij), color=color(1))

    for ax in axs.flatten():
        ax.set_title("")
        ax.set_xlabel("")
        ax.set_ylabel("")

    fig.supxlabel("Frequency [Hz]")

In [ ]:
ref_rcv_id = "obs1"
# obs_ids_visu = obs_ids[[0, 1, 2, -1]]
obs_ids_visu = obs_ids[[0, 1, 2]]
alpha_max = 0.4
# # print(axs.shape)
# for name, node in portion.feature.children.items():
#     ds_feature_section = node.to_dataset()

#     sjj = ds_feature_section.sel(rcv_i=ref_rcv_id, rcv_j=ref_rcv_id).Sxy
#     # Per convention : 2nd index = ref

#     fig, axs = plt.subplots(
#         nrows=obs_ids_visu.size-1, ncols=1, figsize=(16, 16), sharex=True, sharey=True
#     )
#     axs = axs.flatten()
#     iplot = 0
#     for i_rcv, rcv_id_i in enumerate(obs_ids_visu):

#         if rcv_id_i == ref_rcv_id:
#             continue
#         # Sii
#         sii = ds_feature_section.sel(rcv_i=rcv_id_i, rcv_j=rcv_id_i).Sxy
#         # Sij
#         sij = ds_feature_section.sel(rcv_i=rcv_id_i, rcv_j=ref_rcv_id).Sxy

#         # RTF
#         pi_ij = sij.values / sjj.values
#         gamma_ij = 20 * np.log10(np.abs(pi_ij))
#         # np.abs(pi_ij).plot(ax=axs[i_rcv, 0])
#         # gamma_ij.plot(x="freq", ax=axs[iplot], cmap="magma", add_colorbar=False)
#         axs[iplot].pcolormesh(sij.freq, sij.src_datetime, gamma_ij, cmap="magma")

#         axs[iplot].set_xlabel("")
#         axs[iplot].set_ylabel("")
#         # axs[iplot].set_title("")
#         axs[iplot].set_title(rcv_id_i.upper(), fontsize=12)
#         iplot += 1

#     fig.supxlabel("Frequency [Hz]")
#     fig.suptitle(
#         f"{ds_feature_section.section_orientation} {ds_feature_section.section_id}"
#     )
#     # plt.xlim(100, 200)
#     plt.show()

#     # # DCF (sxy SCOT)
#     # dcf_ij = sij / np.sqrt(sii * sjj)
#     # np.real(dcf_ij).plot(ax=axs[i_rcv, 1], color=color(0))
#     # np.imag(dcf_ij).plot(ax=axs[i_rcv, 1], color=color(1))

#     # for ax in axs.flatten():
#     #     ax.set_title("")
#     #     ax.set_xlabel("")
#     #     ax.set_ylabel("")

#     # fig.supxlabel("Frequency [Hz]")

for name, node in portion.feature.children.items():
    ds_feature_section = node.to_dataset()

    sjj = ds_feature_section.sel(rcv_i=ref_rcv_id, rcv_j=ref_rcv_id).Sxy
    # Per convention : 2nd index = ref

    # RTF
    fig_rtf, axs_rtf = plt.subplots(
        nrows=obs_ids_visu.size - 1, ncols=1, figsize=(16, 16), sharex=True, sharey=True
    )
    axs_rtf = axs_rtf.flatten()
    # RTF
    fig_dcf, axs_dcf = plt.subplots(
        nrows=obs_ids_visu.size - 1, ncols=2, figsize=(16, 16), sharex=True, sharey=True
    )

    iplot = 0
    for i_rcv, rcv_id_i in enumerate(obs_ids_visu):

        if rcv_id_i == ref_rcv_id:
            continue
        # Sii
        sii = ds_feature_section.sel(rcv_i=rcv_id_i, rcv_j=rcv_id_i).Sxy
        # Sij
        sij = ds_feature_section.sel(rcv_i=rcv_id_i, rcv_j=ref_rcv_id).Sxy

        # RTF
        pi_ij = sij.values / sjj.values
        gamma_ij = 20 * np.log10(np.abs(pi_ij))

        axs_rtf[iplot].pcolormesh(
            sij.freq,
            sij.src_datetime,
            gamma_ij,
            cmap="magma",
            vmin=-alpha_max * np.nanmax(gamma_ij),
            vmax=+alpha_max * np.nanmax(gamma_ij),
        )

        axs_rtf[iplot].set_xlabel("")
        axs_rtf[iplot].set_ylabel("")
        axs_rtf[iplot].set_title(rcv_id_i.upper(), fontsize=12)

        # DCF (sxy SCOT)
        dcf_ij = sij.values / np.sqrt(sii.values * sjj.values)
        axs_dcf[iplot, 0].pcolormesh(
            np.real(dcf_ij),
            cmap="magma",
            vmin=-1.2,
            vmax=1.2,
        )
        axs_dcf[iplot, 1].pcolormesh(
            np.imag(dcf_ij),
            cmap="magma",
            vmin=-1.2,
            vmax=1.2,
        )

        iplot += 1

    fig_rtf.supxlabel("Frequency [Hz]")
    fig_rtf.suptitle(
        f"{ds_feature_section.section_orientation} {ds_feature_section.section_id}"
    )

    # fpath = os.path.join(
    #     ROOT_IMG_PORTION_FEATURES_RTF,
    #     f"rtf_map_{name}.png",
    # )
    # fig_rtf.savefig(fpath)
    # plt.close(fig_rtf)

    fig_dcf.supxlabel("Frequency [Hz]")
    fig_dcf.suptitle(
        f"{ds_feature_section.section_orientation} {ds_feature_section.section_id}"
    )
    axs_dcf[0, 0].set_title("Real part")
    axs_dcf[0, 1].set_title("Imaginary part")

    # fpath = os.path.join(
    #     ROOT_IMG_PORTION_FEATURES_DCF,
    #     f"dcf_map_{name}.png",
    # )
    # fig_dcf.savefig(fpath)
    # plt.close(fig_dcf)

    plt.show()

In [ ]:

im = plt.pcolormesh(
    np.real(dcf_ij),
    cmap="magma",
    # vmin=-alpha_max * np.nanmax(np.imag(dcf_ij)),
    # vmax=+alpha_max * np.nanmax(np.imag(dcf_ij)),
)

plt.colorbar(im)

# Etude des intersections 

In [ ]:
plot_portion_tir_pos(portion_tree=portion)

## Sud/Nord 0 - Ouest/Est 1

In [ ]:
section_1_name = "sn1"
section_2_name = "oe2"
ds_feature_section1 = portion.feature[section_1_name].to_dataset()
ds_feature_section2 = portion.feature[section_2_name].to_dataset()

### CPA 

In [ ]:
dist_12 = get_distance_between_sections(ds_feature_section1, ds_feature_section2)

In [ ]:
im = plt.pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_12,
    cmap="jet",
)
plt.xlabel(f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory")
plt.ylabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)
plt.colorbar(im, label="Distance [m]")

### Distance dans l'espace des données 

In [ ]:
rcv_id_i = "obs2"
ref_rcv_id = "obs1"

fmin = 5
fmax = 50

pi_ij_a, gamma_a, sij_scot_a = derive_section_features(
    ds_feature_section=ds_feature_section1,
    fmin=fmin,
    fmax=fmax,
    rcv_ref=ref_rcv_id,
    rcv_i=rcv_id_i,
)
pi_ij_b, gamma_b, sij_scot_b = derive_section_features(
    ds_feature_section=ds_feature_section2,
    fmin=fmin,
    fmax=fmax,
    rcv_ref=ref_rcv_id,
    rcv_i=rcv_id_i,
)

In [ ]:
features_section1 = dict(pi_ij=pi_ij_a, gamma_ij=gamma_a, sij_scot=sij_scot_a)
features_section2 = dict(pi_ij=pi_ij_b, gamma_ij=gamma_b, sij_scot=sij_scot_b)

dist_theta, dist_theta_gamma, dist_L2_gamma, dist_dcf = (
    derive_feature_distance_between_sections(features_section1, features_section2)
)

In [ ]:
xmin, ymin = find_min_distance_position(dt1=ds_feature_section1.src_datetime, dt2=ds_feature_section2.src_datetime, dist=dist_12)

# Theta with pi
xmin_theta, ymin_theta = find_min_distance_position(
    dt1=ds_feature_section1.src_datetime,
    dt2=ds_feature_section2.src_datetime,
    dist=dist_theta.T,
)

# Theta with gamma
xmin_theta_gamma, ymin_theta_gamma = find_min_distance_position(
    dt1=ds_feature_section1.src_datetime,
    dt2=ds_feature_section2.src_datetime,
    dist=dist_theta_gamma.T,
)

# L2 with gamma
xmin_L2_gamma, ymin_L2_gamma = find_min_distance_position(
    dt1=ds_feature_section1.src_datetime,
    dt2=ds_feature_section2.src_datetime,
    dist=dist_L2_gamma.T,
)

# DCF
xmin_dcf, ymin_dcf = find_min_distance_position(
    dt1=ds_feature_section1.src_datetime,
    dt2=ds_feature_section2.src_datetime,
    dist=dist_dcf.T,
)

#### R vs theta 

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16, 8))
im = axs[0].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_12,
    cmap="Grays_r",
)
plt.colorbar(im, ax=axs[0], label="Distance [m]")
im = axs[1].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_theta_gamma.T,
    cmap="Grays_r",
    vmin=np.nanmin(dist_theta_gamma),
    vmax=np.nanpercentile(dist_theta_gamma, 0.5),
)
plt.colorbar(im, ax=axs[1], label=r"$\theta_{\gamma} (\in [0, 1])$")

axs[0].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[1].scatter(
    xmin_theta_gamma,
    ymin_theta_gamma,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)
axs[1].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)

for ax in axs.flatten():
    ax.legend(loc="upper left", fontsize=14)

fig.supxlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
fig.supylabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)

#### $\theta_{\gamma}$ vs $\theta_{\Pi}$ 

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(16, 12), sharex=True, sharey=True)
im = axs[0].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_theta_gamma.T,
    cmap="Grays_r",
    # vmin=np.nanpercentile(dist_theta_gamma, 0.1),
    vmin=np.nanmin(dist_theta_gamma),
    vmax=np.nanpercentile(dist_theta_gamma, 0.5),
)
plt.colorbar(im, ax=axs[0], label=r"$\theta_{\gamma} (\in [0, 1])$")
im = axs[1].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_theta.T,
    cmap="Grays_r",
    vmin=np.nanmin(dist_theta),
    vmax=np.nanpercentile(dist_theta, 0.5),
)
plt.colorbar(im, ax=axs[1], label=r"$\theta_{\Pi} (\in [0, 1])$")

axs[0].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[0].scatter(
    xmin_theta_gamma,
    ymin_theta_gamma,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)

axs[1].scatter(
    xmin_theta,
    ymin_theta,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)
axs[1].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)

for ax in axs.flatten():
    ax.legend(loc="upper left", fontsize=14)

fig.supxlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
fig.supylabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)

#### $\theta_{\gamma}$ vs $\theta_{\Pi}$ vs $L2_{\gamma}$

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(16, 16), sharex=True, sharey=True)
im = axs[0].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_theta_gamma.T,
    cmap="Grays_r",
    # vmin=np.nanpercentile(dist_theta_gamma, 0.1),
    vmin=np.nanmin(dist_theta_gamma),
    vmax=np.nanpercentile(dist_theta_gamma, 0.5),
)
plt.colorbar(im, ax=axs[0], label=r"$\theta_{\gamma} (\in [0, 1])$")

im = axs[1].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_L2_gamma.T,
    cmap="Grays_r",
    # vmin=np.nanpercentile(dist_L2_gamma, 0.1),
    vmin=np.nanmin(dist_L2_gamma),
    vmax=np.nanpercentile(dist_L2_gamma, 0.5),
)
plt.colorbar(im, ax=axs[1], label=r"$L_{2_{\gamma}} (\in [0, 1])$")

im = axs[2].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_theta.T,
    cmap="Grays_r",
    # vmin=np.nanpercentile(dist_theta, 0.1),
    vmin=np.nanmin(dist_theta),
    vmax=np.nanpercentile(dist_theta, 0.5),
)
plt.colorbar(im, ax=axs[2], label=r"$\theta_{\Pi} (\in [0, 1])$")


axs[0].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[0].scatter(
    xmin_theta_gamma,
    ymin_theta_gamma,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)

axs[1].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[1].scatter(
    xmin_L2_gamma,
    ymin_L2_gamma,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)

axs[2].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[2].scatter(
    xmin_theta,
    ymin_theta,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)


for ax in axs.flatten():
    ax.legend(loc="upper left", fontsize=14)

fig.supxlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
fig.supylabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)

In [ ]:
idx_ymin_theta_gamma = np.abs(ds_feature_section2.src_datetime - ymin_theta_gamma).argmin(dim="src_datetime")
idx_ymin = np.abs(ds_feature_section2.src_datetime - ymin).argmin(dim="src_datetime")
line_theta_gamma = dist_theta_gamma[:, idx_ymin]

# idx_ymin_theta = np.abs(src_lon2.src_datetime - ymin_theta).argmin(dim="src_datetime")
line_theta = dist_theta[:, idx_ymin]

# idx_ymin_L2_gamma = np.abs(src_lon2.src_datetime - ymin_L2_gamma).argmin(dim="src_datetime")
line_L2_gamma = dist_L2_gamma[:, idx_ymin]

plt.figure()
plt.plot(
    ds_feature_section1.src_datetime,
    line_theta_gamma,
    "-o",
    color=color(0),
    label=r"$\theta_{\gamma}$",
)
plt.plot(
    ds_feature_section1.src_datetime,
    line_L2_gamma,
    "-o",
    color=color(1),
    label=r"$L_{2_{\gamma}}$",
)
plt.plot(
    ds_feature_section1.src_datetime,
    line_theta,
    "-o",
    color=color(2),
    label=r"$\theta_{\Pi}$",
)
plt.legend()

In [ ]:
line_theta_gamma = (line_theta_gamma - np.nanmin(line_theta_gamma)) / (np.nanmax(line_theta_gamma) - np.nanmin(line_theta_gamma))
line_theta = (line_theta - np.nanmin(line_theta)) / (np.nanmax(line_theta) - np.nanmin(line_theta))
line_L2_gamma = (line_L2_gamma - np.nanmin(line_L2_gamma)) / (np.nanmax(line_L2_gamma) - np.nanmin(line_L2_gamma))

plt.figure()
plt.plot(
    ds_feature_section1.src_datetime,
    line_theta_gamma,
    "-o",
    color=color(0),
    label=r"$\theta_{\gamma}$",
)
plt.plot(
    ds_feature_section1.src_datetime,
    line_L2_gamma,
    "-o",
    color=color(1),
    label=r"$L_{2_{\gamma}}$",
)
plt.plot(
    ds_feature_section1.src_datetime,
    line_theta,
    "-o",
    color=color(2),
    label=r"$\theta_{\Pi}$",
)
plt.axvline(xmin, color="k", linestyle="--", label="Target")
plt.legend()

Les trois distances présentent un minimum à la position attendue. Comme précédemment observé, le domaine de monotonie est très faible quelle que soit la distance ! 

In [ ]:
line_theta_gamma = (line_theta_gamma - np.nanmin(line_theta_gamma)) / (
    np.nanmax(line_theta_gamma) - np.nanmin(line_theta_gamma)
)
line_theta = (line_theta - np.nanmin(line_theta)) / (
    np.nanmax(line_theta) - np.nanmin(line_theta)
)
line_L2_gamma = (line_L2_gamma - np.nanmin(line_L2_gamma)) / (
    np.nanmax(line_L2_gamma) - np.nanmin(line_L2_gamma)
)

plt.figure()
plt.plot(
    ds_feature_section1.src_datetime,
    line_theta_gamma,
    "-o",
    color=color(0),
    label=r"$\theta_{\gamma}$",
)
plt.plot(
    ds_feature_section1.src_datetime,
    line_L2_gamma,
    "-o",
    color=color(1),
    label=r"$L_{2_{\gamma}}$",
)
plt.plot(
    ds_feature_section1.src_datetime,
    line_theta,
    "-o",
    color=color(2),
    label=r"$\theta_{\Pi}$",
)
plt.axvline(xmin, color="k", linestyle="--", label="Target")
plt.legend()
plt.xlim(xmin - np.timedelta64(60, "s"), xmin + np.timedelta64(60, "s"))

On dénombre seulement une dizaine de points dans le lobe principale. 

Afin de quantifier le déplacement associé à ces variations rapides de la distance dans l'espace des données on peut évaluer la vitesse de la source à l'intersection. 

In [ ]:
from pyproj import Geod

geod = Geod(ellps="WGS84")

src_lon1, src_lat1 = (
    ds_feature_section1.tir_src_lon,
    ds_feature_section1.tir_src_lat,
)
src_lon2, src_lat2 = (
    ds_feature_section2.tir_src_lon,
    ds_feature_section2.tir_src_lat,
)

# Traj 1
lon1_a = src_lon1.values[:-1]
lat1_a = src_lat1.values[:-1]
lon1_b = src_lon1.values[1:]
lat1_b = src_lat1.values[1:]

_, _, dist_11 = geod.inv(lons1=lon1_a, lats1=lat1_a, lons2=lon1_b, lats2=lat1_b)

# Traj 2
lon2_a = src_lon2.values[:-1]
lat2_a = src_lat2.values[:-1]
lon2_b = src_lon2.values[1:]
lat2_b = src_lat2.values[1:]

_, _, dist_22 = geod.inv(lons1=lon2_a, lats1=lat2_a, lons2=lon2_b, lats2=lat2_b)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16, 8))

im = axs[0].plot(src_lon1.src_datetime[:-1], dist_11, color=color(0))
im = axs[1].plot(src_lon2.src_datetime[:-1], dist_22, color=color(1))

axs[0].set_xlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
axs[1].set_xlabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)
plt.ylabel(
    "Distance between consecutive positions"
)

In [ ]:
speed_1 = dist_11 / np.diff(src_lon1.src_datetime.values).astype("timedelta64[s]").astype(float)
speed_2 = dist_22 / np.diff(src_lon2.src_datetime.values).astype("timedelta64[s]").astype(float)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16, 8))

im = axs[0].plot(src_lon1.src_datetime[:-1], speed_1, color=color(0))
im = axs[1].plot(src_lon2.src_datetime[:-1], speed_2, color=color(1))

axs[0].set_xlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
axs[1].set_xlabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)
fig.supylabel("Speed [m/s]")

print(f"Mean speed along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory: {np.nanmean(speed_1):.2f} m/s")
print(f"Mean speed along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory: {np.nanmean(speed_2):.2f} m/s")

La vitesse moyenne de la source est identique sur les deux trajectoires $\approx$ 2.6 m/s. L'intervalle inter-tir est de 3 s, pour 10 tirs (= 30 sec) on a donc un déplacement d'environ 80 m. 

### Comparaison à la méthode DCF 

#### R vs dcf

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(16, 8))
im = axs[0].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_12,
    cmap="Grays_r",
)
plt.colorbar(im, ax=axs[0], label="Distance [m]")
im = axs[1].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_dcf.T,
    cmap="Grays_r",
    vmin=np.nanmin(dist_dcf),
    vmax=np.nanpercentile(dist_dcf, 0.5),
)
plt.colorbar(im, ax=axs[1], label=r"$DCF (\in [0, 1])$")

axs[0].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[1].scatter(
    xmin_dcf,
    ymin_dcf,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)
axs[1].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)

for ax in axs.flatten():
    ax.legend(loc="upper left", fontsize=14)

fig.supxlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
fig.supylabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)

#### $\theta_{\gamma}$ vs DCF 

In [ ]:
# vmin = max(np.nanmin(dist_theta_gamma), np.nanmin(dist_dcf))
# vmax = min(np.nanpercentile(dist_theta_gamma, 0.75), np.nanpercentile(dist_dcf, 0.75))


fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(16, 12), sharex=True, sharey=True)
im = axs[0].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_theta_gamma.T,
    cmap="Grays_r",
    # vmin=np.nanpercentile(dist_theta_gamma, 0.1),
    vmin=np.nanmin(dist_theta_gamma),
    vmax=np.nanpercentile(dist_theta_gamma, 0.99),
    # vmin=vmin,
    # vmax=vmax,
)
plt.colorbar(im, ax=axs[0], label=r"$\theta_{\gamma} (\in [0, 1])$")
im = axs[1].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_dcf.T,
    cmap="Grays_r",
    vmin=np.nanmin(dist_dcf),
    vmax=np.nanpercentile(dist_dcf, 0.99),
    # vmin=vmin,
    # vmax=vmax,
)
plt.colorbar(im, ax=axs[1], label=r"$DCF (\in [0, 1])$")

# Common colorbar
# plt.colorbar(im, ax=axs, label="Distance")

axs[0].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[0].scatter(
    xmin_theta_gamma,
    ymin_theta_gamma,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)

axs[1].scatter(
    xmin_dcf,
    ymin_dcf,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)
axs[1].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)

for ax in axs.flatten():
    ax.legend(loc="upper left", fontsize=14)

fig.supxlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
fig.supylabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)

Afin de comparer les deux approches on normalise afin d'avoir des variations dans la pleine échelle [0, 1]

In [ ]:
dist_dcf_norm = (dist_dcf - np.nanmin(dist_dcf)) / (np.nanmax(dist_dcf) - np.nanmin(dist_dcf))
dist_theta_gamma_norm = (dist_theta_gamma - np.nanmin(dist_theta_gamma)) / (np.nanmax(dist_theta_gamma) - np.nanmin(dist_theta_gamma))

In [ ]:
# vmin = max(np.nanmin(dist_theta_gamma), np.nanmin(dist_dcf))
# vmax = min(np.nanpercentile(dist_theta_gamma, 0.5), np.nanpercentile(dist_dcf, 0.5))

vmin = 0
vmax = 1

fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(16, 12), sharex=True, sharey=True)
im = axs[0].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_theta_gamma_norm.T,
    cmap="Grays_r",
    # vmin=np.nanpercentile(dist_theta_gamma, 0.1),
    # vmin=np.nanmin(dist_theta_gamma),
    # vmax=np.nanpercentile(dist_theta_gamma, 0.5),
    vmin=vmin,
    vmax=vmax,
)
# plt.colorbar(im, ax=axs[0], label=r"$\theta_{\gamma} (\in [0, 1])$")
im = axs[1].pcolormesh(
    ds_feature_section1.src_datetime,
    ds_feature_section2.src_datetime,
    dist_dcf_norm.T,
    cmap="Grays_r",
    # vmin=np.nanmin(dist_dcf),
    # vmax=np.nanpercentile(dist_dcf, 0.5),
    vmin=vmin,
    vmax=vmax,
)
# plt.colorbar(im, ax=axs[1], label=r"$DCF (\in [0, 1])$")

# Common colorbar
plt.colorbar(im, ax=axs, label="Distance")

axs[0].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)
axs[0].scatter(
    xmin_theta_gamma,
    ymin_theta_gamma,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)

axs[1].scatter(
    xmin_dcf,
    ymin_dcf,
    marker="o",
    s=200,
    facecolors="none",
    edgecolors="m",
    label="Guess",
    linewidth=3,
)
axs[1].scatter(
    xmin,
    ymin,
    marker="s",
    s=400,
    facecolors="none",
    edgecolors="r",
    label="Target",
    linewidth=3,
)

for ax in axs.flatten():
    ax.legend(loc="upper left", fontsize=14)

fig.supxlabel(
    f"Time along {ds_feature_section1.section_orientation} {ds_feature_section1.section_id} trajectory"
)
fig.supylabel(
    f"Time along {ds_feature_section2.section_orientation} {ds_feature_section2.section_id} trajectory"
)

In [ ]:
idx_ymin = np.abs(ds_feature_section2.src_datetime - ymin_theta_gamma).argmin(
    dim="src_datetime"
)
line_theta_gamma_norm = dist_theta_gamma_norm[:, idx_ymin]
idx_ymin = np.abs(ds_feature_section2.src_datetime - ymin_dcf).argmin(
    dim="src_datetime"
)
line_dcf_norm = dist_dcf_norm[:, idx_ymin]

plt.figure()
plt.plot(
    ds_feature_section1.src_datetime,
    line_theta_gamma_norm,
    "-o",
    color=color(0),
    label=r"$\theta_{\gamma}$",
)
plt.plot(
    ds_feature_section1.src_datetime,
    line_dcf_norm,
    "-o",
    color=color(1),
    label=r"$DCF$",
)

plt.legend()

Ce n'est pas une bonne manière de comparer les deux résultats : dans le cas où l'un des deux OBS du couple est distant de l'OBS de référence theta croît avec la distance au point d'intersection et la normalisation dépend donc de la taille du domaine que l'on regarde. Plus le domaine est grand plus le contraste au point d'intersection va dimuner par normalisation.